# 01 - Khám Phá Dữ Liệu (EDA)

**Mục tiêu:**
- Mô tả dữ liệu và data dictionary
- Thống kê mô tả, phân phối
- Phân tích missing values
- Phân tích tương quan
- Phân tích target

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1.1 Load dữ liệu

In [ ]:
from src.data.loader import load_raw_data, check_schema, get_data_summary, print_data_dictionary

df = load_raw_data()

In [ ]:
# Data Dictionary
print_data_dictionary()

In [ ]:
# Kiểm tra schema
schema = check_schema(df)

In [ ]:
# Xem mẫu dữ liệu
df.head(10)

In [ ]:
# Thông tin tổng quan
df.info()

## 1.2 Thống kê mô tả

In [ ]:
# Bảng summary
summary = get_data_summary(df)
summary

In [ ]:
# Thống kê mô tả cho biến số
df.describe().round(2)

In [ ]:
# Thống kê cho biến phân loại
df.describe(include='object')

## 1.3 Phân tích Missing Values

In [ ]:
from src.visualization.plots import plot_missing_values

fig = plot_missing_values(df, save=True)
plt.show()

In [ ]:
# Bảng missing chi tiết
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Count': missing, 'Percent': missing_pct})
missing_df = missing_df[missing_df['Count'] > 0].sort_values('Percent', ascending=False)
missing_df

## 1.4 Phân phối Target

In [ ]:
# Phân phối target gốc (multi-class 0-4)
print("Target distribution (original - num):")
print(df['num'].value_counts().sort_index())
print(f"\nBinary: No Disease={( df['num']==0).sum()}, Disease={(df['num']>0).sum()}")

In [ ]:
# Tạo binary target để vẽ
df_temp = df.copy()
df_temp['target'] = (df_temp['num'] > 0).astype(int)

from src.visualization.plots import plot_target_distribution
fig = plot_target_distribution(df_temp, save=True)
plt.show()

## 1.5 Phân phối các biến số

In [ ]:
from src.visualization.plots import plot_numeric_distributions

numeric_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
fig = plot_numeric_distributions(df, cols=numeric_cols, save=True)
plt.show()

## 1.6 Phân phối biến phân loại

In [ ]:
cat_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'thal']

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(cat_cols):
    if i < len(axes):
        df[col].value_counts().plot(kind='bar', ax=axes[i], color=sns.color_palette('husl', df[col].nunique()))
        axes[i].set_title(f'{col}')
        axes[i].tick_params(axis='x', rotation=45)

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/categorical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.7 Ma trận tương quan

In [ ]:
# Chỉ dùng biến số cho correlation
numeric_df = df.select_dtypes(include='number')

from src.visualization.plots import plot_correlation_matrix
fig = plot_correlation_matrix(numeric_df, save=True)
plt.show()

## 1.8 Biến số theo Target

In [ ]:
df_temp = df.copy()
df_temp['target'] = (df_temp['num'] > 0).astype(int)

from src.visualization.plots import plot_features_vs_target
fig = plot_features_vs_target(df_temp, cols=['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca'], save=True)
plt.show()

## 1.9 Biến phân loại theo Target

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, col in enumerate(['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope']):
    ct = pd.crosstab(df[col], df_temp['target'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'])
    axes[i].set_title(f'{col} vs Target')
    axes[i].legend(['No Disease', 'Disease'])
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Categorical Features vs Target', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/figures/categorical_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.10 Rủi ro & Thách thức

> ⚠️ **Tiêu chí A yêu cầu nêu rủi ro dữ liệu**

### Mất cân bằng lớp (Class Imbalance)
- 411 không bệnh (45%) vs 509 bệnh (55%) → nhẹ, nhưng cần xử lý
- Giải pháp: SMOTE + class_weight='balanced'

### Missing Values nghiêm trọng
- `ca`: 66%, `thal`: 53%, `slope`: 34% → cần chiến lược fill hợp lý
- Drop cột missing > 40% hoặc fill median/mode

### Data Leakage tiềm ẩn
- PHẢI scale SAU khi split train/test
- SMOTE chỉ áp dụng trên train set
- Cross-validation dùng StratifiedKFold để giữ tỷ lệ lớp

### Dữ liệu đa nguồn
- 4 trung tâm y tế khác nhau → phân phối có thể khác biệt
- Cột `dataset` bị drop để tránh bias theo nguồn

In [ ]:
# Phân tích theo nguồn dữ liệu
print("Phân phối target theo nguồn dữ liệu:")
ct = pd.crosstab(df['dataset'], df_temp['target'], normalize='index').round(3)
print(ct)
print("\n→ Tỷ lệ bệnh khác nhau giữa các nguồn: cần cẩn thận khi tổng quát hoá")

## 1.11 Nhận xét EDA

**Dữ liệu:**
- 920 mẫu, 16 cột (14 features + id + dataset)
- Missing values cao ở `ca` (66%), `thal` (53%), `slope` (34%)
- Target imbalanced nhẹ: 411 no disease vs 509 disease

**Phát hiện chính:**
- Cần xử lý missing values trước khi modeling
- Các biến `cp`, `thalch`, `oldpeak`, `ca` có tương quan mạnh với target
- Cần encoding biến phân loại
- Dữ liệu gộp từ 4 nguồn với tỷ lệ bệnh khác nhau